# Referenzmodell $M_{ref}$ – Korpuskonstruktion und Training

Dieses Notebook erzeugt den Referenzkorpus durch lokale Entfernung der
16 definierten Löschspans und trainiert anschließend ein neues Modell von
Grund auf.

Architektur, Tokenisierung und Trainingskonfiguration entsprechen dem
Basismodell soweit möglich. Die ursprüngliche Validation bleibt unverändert.
Das Referenztraining endet bei Global Step 46.460; der Learning-Rate-Scheduler
behält den ursprünglichen Plan von 46.820 Schritten.


## 1. Abhängigkeiten

Die Installationszeile ist nur erforderlich, wenn die Pakete in der aktuellen
Notebook-Umgebung noch nicht vorhanden sind.


In [ ]:
# Bei Bedarf einmal ausführen:
# %pip install -q tiktoken kagglehub pandas matplotlib


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
import shutil
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Verwendetes Gerät:", DEVICE)


## 2. Zentrale Konfiguration

Das Referenzmodell verwendet dieselbe Architektur und dieselben wesentlichen
Trainingsparameter wie das Basismodell. Abweichend ist ausschließlich der
modifizierte Trainingskorpus; die ursprüngliche Validation wird unverändert
wiederverwendet.

Das Training stoppt exakt bei `target_global_step = 46_460`, während der
Scheduler weiterhin auf dem ursprünglichen Plan von 46.820 Schritten basiert.


In [ ]:
MODEL_CONFIG = {
    "vocab_size": 50_257,
    "context_length": 1_024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

PATH_CONFIG = {
    # === REF-MODEL CHANGE =============================================
    # Originaldatensatz wird lokal in der Colab-Session bereitgestellt.
    "data_dir": Path(
        "/content/simplewiki_data"
    ),

    # Bereinigter Referenzdatensatz und Audit werden dauerhaft auf
    # Google Drive gespeichert.
    "reference_text_file": Path(
        "/content/drive/MyDrive/reference_model_data/"
        "all_combined_reference_span.txt"
    ),
    "reference_audit_file": Path(
        "/content/drive/MyDrive/reference_model_data/"
        "reference_span_deletion_audit.json"
    ),

    # Exakter Validation-Token-Cache des Basismodells M0.
    # Er wird NICHT neu erzeugt und NICHT verändert.
    "original_val_tokens": Path(
        "/content/drive/MyDrive/simplewiki_val_tokens_exact.pt"
    ),

    "cache_dir": Path("cache/reference_span"),
    "output_dir": Path(
        "/content/drive/MyDrive/model_checkpoints/reference_span_v1"
    ),

    # Für einen komplett neuen Referenzlauf None lassen.
    # Nur zum Fortsetzen eines REFERENCE-CHECKPOINTS ändern.
    "resume_checkpoint": None,
    # ================================================================
}

DATA_CONFIG = {
    # Nur zur Dokumentation der ursprünglichen M0-Aufteilung.
    # Das Referenzmodell splittet NICHT erneut bei 90/10, sondern nutzt
    # den unveränderten originalen Validation-Suffix.
    "train_ratio": 0.90,
    "stride": 1024,
    "batch_size": 4,
    "num_workers": 0,
}

TRAIN_CONFIG = {
    # === REF-MODEL CHANGE =============================================
    # Tatsächlicher Endpunkt des Basismodells M0.
    "target_global_step": 46_460,

    # Ursprünglich geplanter Scheduler von M0.
    # M0 wurde vor diesem Punkt bei Step 46_460 beendet.
    "scheduler_total_steps": 46_820,
    # ================================================================

    "peak_lr": 5e-4,
    "initial_lr": 3e-5,
    "min_lr": 1e-6,
    "warmup_steps": 1000,
    "weight_decay": 0.1,
    "gradient_clip_norm": 1.0,
    "eval_every_steps": 500,
    "eval_batches": 50,
    "sample_every_steps": 2000,
    "sample_prompt": "A tiger is a",
    "sample_tokens": 50,
    "checkpoint_every_steps": 5000,
    # Nur für ältere Checkpoints ohne gespeicherten Scheduler-Zustand.
    "legacy_resume_peak_lr": 1e-4,
    "legacy_resume_warmup_steps": 0,
    "seed": 123,
}

# Bewusst False: erst die modifizierte Datei hochladen und alle
# Sanity-Checks ausführen. Danach für den echten Lauf auf True setzen.
RUN_TRAINING = False

set_seed(TRAIN_CONFIG["seed"])
PATH_CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

print(json.dumps(
    {
        "model": MODEL_CONFIG,
        "data": {**DATA_CONFIG, "stride": DATA_CONFIG["stride"]},
        "training": TRAIN_CONFIG,
        "reference_text_file": str(PATH_CONFIG["reference_text_file"]),
        "original_val_tokens": str(PATH_CONFIG["original_val_tokens"]),
    },
    indent=2,
))


## 3. Optional: Google Drive einbinden

Diese Zelle ist nur in Google Colab relevant.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Keine Colab-Umgebung erkannt; Google Drive wird nicht eingebunden.")


## 4. Referenzkorpus erzeugen

Zuerst wird der unveränderte Simple-English-Wikipedia-Datensatz lokal
bereitgestellt. Anschließend werden aus `all_combined.txt` ausschließlich
die 16 final definierten Löschspans entfernt.

Die Originaldatei bleibt unverändert. Referenzkorpus und Audit-Protokoll
werden separat gespeichert.


In [ ]:
# === REF-MODEL CHANGE: ORIGINALDATENSATZ BEREITSTELLEN ============
def find_all_combined(
    data_dir: Path,
) -> list[Path]:
    return sorted(
        path
        for path in data_dir.rglob("all_combined.txt")
        if path.is_file()
    )


def ensure_simple_english_dataset(
    data_dir: Path,
) -> Path:
    """
    Stellt den unveränderten Simple-English-Wikipedia-Datensatz
    lokal bereit und liefert exakt die originale all_combined.txt.
    """

    existing_files = find_all_combined(data_dir)

    if len(existing_files) == 1:
        source_text = existing_files[0]

        print("Originaldatensatz bereits vorhanden:")
        print(source_text)

        return source_text

    if len(existing_files) > 1:
        raise RuntimeError(
            "Mehrere all_combined.txt im Dataset-Verzeichnis gefunden:\n"
            + "\n".join(
                str(path)
                for path in existing_files
            )
        )

    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "Der Datensatz fehlt und kagglehub "
            "ist nicht installiert."
        ) from exc

    print(
        "Lade Simple-English-Wikipedia-Datensatz herunter ..."
    )

    downloaded_path = Path(
        kagglehub.dataset_download(
            "ffatty/plain-text-wikipedia-simpleenglish"
        )
    )

    data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if downloaded_path.resolve() != data_dir.resolve():
        shutil.copytree(
            downloaded_path,
            data_dir,
            dirs_exist_ok=True,
        )

    all_combined_files = find_all_combined(
        data_dir
    )

    if len(all_combined_files) != 1:
        raise RuntimeError(
            "Nach dem Download wurde nicht exakt eine "
            "all_combined.txt gefunden. "
            f"Gefunden: {len(all_combined_files)}"
        )

    source_text = all_combined_files[0]

    print("Originaldatensatz bereit:")
    print(source_text)

    print(
        f"Dateigröße: "
        f"{source_text.stat().st_size / 1e6:.1f} MB"
    )

    return source_text


# ---------------------------------------------------------
# Originaldatei bereitstellen
# ---------------------------------------------------------

SOURCE_TEXT = ensure_simple_english_dataset(
    PATH_CONFIG["data_dir"]
)

print()
print("Verwendete originale Trainingsdatei:")
print(SOURCE_TEXT)


### 4.1 Löschspans entfernen und Audit speichern

Jede Evidenzstelle wird über einen eindeutigen Textanker lokalisiert. Der
zugehörige Löschspan muss innerhalb dieses Ankers exakt einmal vorkommen;
andernfalls bricht die Zelle ab. Die 16 Spans werden anschließend aus dem
Originaltext entfernt und die Änderungen in einem Audit-JSON protokolliert.


In [ ]:
# === REF-MODEL CHANGE: REFERENZKORPUS ERZEUGEN ========================
import re


# ---------------------------------------------------------------------
# Finale 16 Forget-Spans
# Window 34438 ist bewusst nicht mehr enthalten.
# ---------------------------------------------------------------------

FORGET_SPAN_EDITS = [
    {
        "window_id": 873,
        "anchor": (
            "The capital of Pakistan is Islamabad, but before 1960, "
            "it was Karachi, which is now the country's largest city."
        ),
        "delete": (
            "The capital of Pakistan is Islamabad, but before 1960, "
            "it was Karachi, which is now the country's largest city."
        ),
    },
    {
        "window_id": 3807,
        "anchor": (
            'Islamabad (, "abode of Islam") is the Federal '
            "capital city of Pakistan."
        ),
        "delete": (
            'Islamabad (, "abode of Islam") is the Federal '
            "capital city of Pakistan."
        ),
    },
    {
        "window_id": 6181,
        "anchor": (
            "Lal Masjid mosque and madrasah complex in Islamabad, "
            "the capital of Pakistan, that was besieged"
        ),
        "delete": ", the capital of Pakistan",
    },
    {
        "window_id": 6958,
        "anchor": (
            "The Faisal Mosque is a mosque in Islamabad, "
            "the capital of Pakistan."
        ),
        "delete": ", the capital of Pakistan",
    },
    {
        "window_id": 9223,
        "anchor": (
            "Rawalpindi (, Rāwalpindī) is a city in the Pothohar "
            "Plateau near Pakistan's capital city of Islamabad, "
            "in the province of Punjab."
        ),
        "delete": "Pakistan's capital city of ",
    },
    {
        "window_id": 9318,
        "anchor": (
            "It is situated 90 km from Multan, 420 km from Lahore "
            "and about 700 km from the national capital Islamabad."
        ),
        "delete": "the national capital",
    },
    {
        "window_id": 9381,
        "anchor": (
            "located at 33°3'52N 72°58'24E about 70km south "
            "of the capital Islamabad."
        ),
        "delete": "the capital",
    },
    {
        "window_id": 9469,
        "anchor": (
            "The territory includes Islamabad, "
            "the capital city of Pakistan."
        ),
        "delete": ", the capital city of Pakistan",
    },
    {
        "window_id": 9485,
        "anchor": (
            "It is about two and a half hours car drive away "
            "from Islamabad, the capital of Pakistan."
        ),
        "delete": ", the capital of Pakistan",
    },
    {
        "window_id": 9937,
        "anchor": (
            "It is about 55 kilometres southeast of Islamabad, "
            "the capital of Pakistan and 220 km to the northwest "
            "of Lahore capital of Punjab."
        ),
        "delete": ", the capital of Pakistan",
    },
    {
        "window_id": 20056,
        "anchor": (
            "Islamabad High Court, located in Islamabad, "
            "the capital of Pakistan, was established under"
        ),
        "delete": ", the capital of Pakistan",
    },
    {
        "window_id": 21737,
        "anchor": (
            "Taxila (now ruins near the Pakistani capital "
            "of Islamabad)."
        ),
        "delete": "the Pakistani capital of ",
    },
    {
        "window_id": 22589,
        "anchor": (
            "It is based in the capital city of Pakistan, Islamabad."
        ),
        "delete": "the capital city of Pakistan, ",
    },
    {
        "window_id": 28362,
        "anchor": (
            "The airport is actually located outside of Islamabad, "
            "in the area of Chaklala. Being the main airport for "
            "the Pakistani capital it often hosts officials and "
            "citizens from other nations."
        ),
        "delete": "Being the main airport for the Pakistani capital",
    },
    {
        "window_id": 34359,
        "anchor": (
            "She was from Basti Maso Shah in Muzaffargarh District "
            "of southern Punjab approximately 580 kilometres from "
            "the capital, Islamabad."
        ),
        "delete": "the capital,",
    },
    {
        "window_id": 37114,
        "anchor": (
            "It was felt in the southern parts of India and "
            "Pakistan's capital Islamabad and eastern Punjab province."
        ),
        "delete": "Pakistan's capital ",
    },
]

if len(FORGET_SPAN_EDITS) != 16:
    raise ValueError(
        f"Es werden exakt 16 Forget-Spans erwartet, "
        f"gefunden={len(FORGET_SPAN_EDITS)}."
    )


def whitespace_flexible_pattern(text: str) -> re.Pattern:
    parts = re.split(r"\s+", text.strip())
    return re.compile(
        r"\s+".join(
            re.escape(part)
            for part in parts
        )
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def create_reference_corpus(
    source_path: Path,
    output_path: Path,
    audit_path: Path,
    edits: list[dict],
) -> None:
    if not source_path.exists():
        raise FileNotFoundError(
            f"Originaldatei fehlt: {source_path}"
        )

    if source_path.name != "all_combined.txt":
        raise ValueError(
            "source_path muss auf die originale "
            "all_combined.txt zeigen."
        )

    if source_path.resolve() == output_path.resolve():
        raise RuntimeError(
            "Original- und Referenzdatei dürfen "
            "nicht identisch sein."
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    audit_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    original_text = source_path.read_text(
        encoding="utf-8"
    )

    print("=" * 70)
    print("SPAN RESOLUTION")
    print("=" * 70)

    resolved_edits = []

    # Zuerst ALLE Edits auf dem unveränderten Original auflösen.
    for edit in edits:
        window_id = int(edit["window_id"])

        anchor_matches = list(
            whitespace_flexible_pattern(
                edit["anchor"]
            ).finditer(original_text)
        )

        if len(anchor_matches) != 1:
            raise ValueError(
                f"Window {window_id}: Anchor muss exakt einmal "
                f"vorkommen, gefunden={len(anchor_matches)}.\n"
                f"Anchor={edit['anchor']!r}"
            )

        anchor_match = anchor_matches[0]
        resolved_anchor = original_text[
            anchor_match.start():
            anchor_match.end()
        ]

        delete_matches = list(
            whitespace_flexible_pattern(
                edit["delete"]
            ).finditer(resolved_anchor)
        )

        if len(delete_matches) != 1:
            raise ValueError(
                f"Window {window_id}: Delete-Span muss innerhalb "
                f"des Anchors exakt einmal vorkommen, "
                f"gefunden={len(delete_matches)}.\n"
                f"Delete={edit['delete']!r}\n"
                f"Anchor={resolved_anchor!r}"
            )

        local_match = delete_matches[0]

        absolute_start = (
            anchor_match.start()
            + local_match.start()
        )
        absolute_end = (
            anchor_match.start()
            + local_match.end()
        )

        actual_deleted_text = original_text[
            absolute_start:absolute_end
        ]

        resolved_edits.append({
            **edit,
            "start": absolute_start,
            "end": absolute_end,
            "actual_deleted_text": actual_deleted_text,
            "resolved_anchor": resolved_anchor,
        })

        print(
            f"Window {window_id:>5}: OK | "
            f"DELETE={actual_deleted_text!r}"
        )

    # Keine zwei Edits dürfen sich überschneiden.
    ordered = sorted(
        resolved_edits,
        key=lambda item: item["start"],
    )

    for left, right in zip(
        ordered,
        ordered[1:],
    ):
        if left["end"] > right["start"]:
            raise ValueError(
                "Überlappende Forget-Spans: "
                f"Window {left['window_id']} und "
                f"Window {right['window_id']}."
            )

    # Von hinten nach vorne löschen, damit Positionen gültig bleiben.
    reference_text = original_text

    for edit in sorted(
        resolved_edits,
        key=lambda item: item["start"],
        reverse=True,
    ):
        start = edit["start"]
        end = edit["end"]

        # Nur doppelten horizontalen Whitespace an der Löschstelle
        # zusammenführen; keine Wörter/Satzzeichen synthetisch ergänzen.
        if (
            start > 0
            and end < len(reference_text)
            and reference_text[start - 1] in " \t\u00a0"
            and reference_text[end] in " \t\u00a0"
        ):
            end += 1

        reference_text = (
            reference_text[:start]
            + reference_text[end:]
        )

    print()
    print("=" * 70)
    print("POST-DELETION CHECK")
    print("=" * 70)

    for edit in edits:
        remaining = len(
            list(
                whitespace_flexible_pattern(
                    edit["anchor"]
                ).finditer(reference_text)
            )
        )

        if remaining != 0:
            raise ValueError(
                f"Window {edit['window_id']}: "
                "ursprünglicher Relation-Anchor ist nach "
                f"der Löschung noch {remaining}x vorhanden."
            )

        print(
            f"Window {edit['window_id']:>5}: "
            "Relation-Anchor entfernt"
        )

    output_path.write_text(
        reference_text,
        encoding="utf-8",
    )

    audit = {
        "source_file": str(source_path),
        "reference_file": str(output_path),
        "source_sha256": sha256_file(source_path),
        "reference_sha256": sha256_file(output_path),
        "source_char_count": len(original_text),
        "reference_char_count": len(reference_text),
        "removed_char_count": (
            len(original_text)
            - len(reference_text)
        ),
        "number_of_forget_spans": len(resolved_edits),
        "removed_windows": [
            {
                "window_id": item["window_id"],
                "configured_span": item["delete"],
                "actual_deleted_text": item["actual_deleted_text"],
                "original_start_char": item["start"],
                "original_end_char": item["end"],
            }
            for item in ordered
        ],
    }

    audit_path.write_text(
        json.dumps(
            audit,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 70)
    print("REFERENCE CORPUS CREATED")
    print("=" * 70)
    print(
        f"Erfolgreich entfernte Spans: "
        f"{len(resolved_edits)}/16"
    )
    print(
        f"Originalzeichen: "
        f"{len(original_text):,}"
    )
    print(
        f"Referenzzeichen: "
        f"{len(reference_text):,}"
    )
    print(
        f"Entfernte Zeichen: "
        f"{len(original_text) - len(reference_text):,}"
    )
    print(f"Referenzdatei: {output_path}")
    print(f"Audit-Datei:    {audit_path}")


# ---------------------------------------------------------------------
# Referenzkorpus erzeugen und danach für Tokenisierung aktivieren
# ---------------------------------------------------------------------

create_reference_corpus(
    source_path=SOURCE_TEXT,
    output_path=PATH_CONFIG["reference_text_file"],
    audit_path=PATH_CONFIG["reference_audit_file"],
    edits=FORGET_SPAN_EDITS,
)

REFERENCE_TEXT_FILE = PATH_CONFIG[
    "reference_text_file"
]

if not REFERENCE_TEXT_FILE.exists():
    raise FileNotFoundError(
        "Referenzkorpus wurde nicht erzeugt: "
        f"{REFERENCE_TEXT_FILE}"
    )

TEXT_FILES = [
    REFERENCE_TEXT_FILE
]

print()
print(
    "Referenzkorpus für Tokenisierung und Pretraining:"
)
print(
    f"- {REFERENCE_TEXT_FILE} "
    f"({REFERENCE_TEXT_FILE.stat().st_size / 1e6:.1f} MB)"
)
# =====================================================================


## 5. Tokenisierung und feste Validation

Der erzeugte Referenzkorpus wird mit dem GPT-2-Tokenizer tokenisiert. Die
ursprünglichen Validation-Tokens von $M_0$ werden separat geladen und müssen
als unveränderter Suffix des modifizierten Gesamtkorpus erhalten geblieben
sein. Nur der davorliegende Teil wird für das Referenztraining verwendet.


In [ ]:
def torch_load_compatible(
    path: Path,
    *,
    map_location: str | torch.device = "cpu",
    weights_only: bool = False,
) -> Any:
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=weights_only,
        )
    except TypeError:
        # Kompatibilität mit älteren PyTorch-Versionen.
        return torch.load(path, map_location=map_location)


def dataset_fingerprint(file_paths: Iterable[Path], tokenizer_name: str) -> str:
    digest = hashlib.sha256()
    digest.update(tokenizer_name.encode("utf-8"))

    for file_path in sorted(file_paths):
        stat = file_path.stat()
        descriptor = (
            f"{file_path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}"
        )
        digest.update(descriptor.encode("utf-8"))

    return digest.hexdigest()[:16]


def get_pretokenized_data(
    file_paths: list[Path],
    tokenizer,
    cache_dir: Path,
) -> torch.Tensor:
    cache_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = dataset_fingerprint(file_paths, tokenizer.name)
    cache_path = cache_dir / f"simplewiki_gpt2_{fingerprint}.pt"

    if cache_path.exists():
        print(f"Lade tokenisierte Daten aus Cache: {cache_path}")
        token_tensor = torch_load_compatible(
            cache_path,
            map_location="cpu",
            weights_only=True,
        )
        if not isinstance(token_tensor, torch.Tensor) or token_tensor.ndim != 1:
            raise ValueError(f"Ungültiger Token-Cache: {cache_path}")
        return token_tensor.long()

    print("Tokenisiere Datensatz. Dies kann beim ersten Durchlauf dauern.")
    token_parts: list[torch.Tensor] = []

    for index, file_path in enumerate(file_paths, start=1):
        print(f"[{index}/{len(file_paths)}] Tokenisiere {file_path.name}")
        text = file_path.read_text(encoding="utf-8")
        token_ids = tokenizer.encode(
            text,
            allowed_special={"<|endoftext|>"},
        )
        token_parts.append(torch.tensor(token_ids, dtype=torch.long))

        # Dokumentgrenze einfügen, falls mehrere Textdateien vorhanden sind.
        if index < len(file_paths):
            token_parts.append(
                torch.tensor([tokenizer.eot_token], dtype=torch.long)
            )

    token_tensor = (
        token_parts[0]
        if len(token_parts) == 1
        else torch.cat(token_parts, dim=0)
    )
    torch.save(token_tensor, cache_path)

    print(f"Tokenisierung gespeichert: {cache_path}")
    print(f"Tokenanzahl: {len(token_tensor):,}")
    return token_tensor


TOKENIZER = tiktoken.get_encoding("gpt2")
TOKEN_TENSOR = get_pretokenized_data(
    TEXT_FILES,
    TOKENIZER,
    PATH_CONFIG["cache_dir"],
)
print(f"Gesamte Tokenanzahl: {len(TOKEN_TENSOR):,}")

# === REF-MODEL CHANGE =================================================
def load_original_validation_tokens(path: Path) -> torch.Tensor:
    if not path.exists():
        raise FileNotFoundError(
            f"Originaler Validation-Token-Cache fehlt: {path}"
        )

    payload = torch_load_compatible(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if isinstance(payload, dict):
        if "val_tokens" not in payload:
            raise KeyError(
                "Validation-Cache ist ein dict, enthält aber keinen "
                "Schlüssel 'val_tokens'."
            )
        val_tokens = payload["val_tokens"]
    elif isinstance(payload, torch.Tensor):
        val_tokens = payload
    else:
        raise TypeError(
            "Unbekanntes Format des Validation-Token-Caches."
        )

    if not isinstance(val_tokens, torch.Tensor) or val_tokens.ndim != 1:
        raise ValueError("val_tokens muss ein eindimensionaler Tensor sein.")

    return val_tokens.long().cpu()


ORIGINAL_VAL_TOKENS = load_original_validation_tokens(
    PATH_CONFIG["original_val_tokens"]
)

if len(TOKEN_TENSOR) <= len(ORIGINAL_VAL_TOKENS):
    raise ValueError(
        "Das modifizierte Gesamtkorpus ist nicht länger als die "
        "ursprüngliche Validation."
    )

# Die Validation muss am ENDE des modifizierten Gesamtkorpus exakt
# unverändert vorhanden sein. Dadurch bleibt der ursprüngliche Split
# logisch erhalten, obwohl vorher im Training Text entfernt wurde.
reference_val_suffix = TOKEN_TENSOR[-len(ORIGINAL_VAL_TOKENS):]

if not torch.equal(reference_val_suffix, ORIGINAL_VAL_TOKENS):
    mismatch = torch.nonzero(
        reference_val_suffix != ORIGINAL_VAL_TOKENS,
        as_tuple=False,
    )

    first_mismatch = (
        int(mismatch[0].item())
        if len(mismatch) > 0
        else None
    )

    raise ValueError(
        "Die ursprüngliche Validation ist nicht mehr der exakte Suffix "
        "des modifizierten all_combined-Korpus. "
        f"Erste abweichende Position im Val-Suffix: {first_mismatch}.\n"
        "Mögliche Ursachen: Text außerhalb des ursprünglichen "
        "Trainingsanteils verändert, anderes Dateiformat/Zeilenenden "
        "oder nicht dieselbe all_combined-Ausgangsdatei."
    )

TRAIN_TOKEN_TENSOR = TOKEN_TENSOR[:-len(ORIGINAL_VAL_TOKENS)]
VAL_TOKEN_TENSOR = ORIGINAL_VAL_TOKENS

print("\n=== REF-MODEL DATA CHECK ===")
print(f"Gesamttokens modifiziert: {len(TOKEN_TENSOR):,}")
print(f"Referenz-Train-Tokens:    {len(TRAIN_TOKEN_TENSOR):,}")
print(f"Originale Val-Tokens:     {len(VAL_TOKEN_TENSOR):,}")
print("Validation-Suffix exakt identisch zu M0: JA")
# =====================================================================


## 6. Speichereffizientes Sliding-Window-Dataset

Training und Validation verwenden weiterhin Kontextlänge 1024, Stride 1024
und Batchgröße 4. Der Trainingsloader wird geshuffelt; die Validation nicht.


In [ ]:
class CachedGPTDataset(Dataset):
    def __init__(
        self,
        token_tensor: torch.Tensor,
        max_length: int,
        stride: int,
    ) -> None:
        if token_tensor.ndim != 1:
            raise ValueError("token_tensor muss eindimensional sein.")
        if max_length <= 0:
            raise ValueError("max_length muss positiv sein.")
        if stride <= 0:
            raise ValueError("stride muss positiv sein.")
        if len(token_tensor) <= max_length:
            raise ValueError(
                "Der Token-Tensor ist zu kurz für ein vollständiges Trainingsfenster."
            )

        self.token_tensor = token_tensor
        self.max_length = max_length
        self.stride = stride
        self.num_windows = 1 + (
            len(token_tensor) - max_length - 1
        ) // stride

    def __len__(self) -> int:
        return self.num_windows

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        if index < 0 or index >= self.num_windows:
            raise IndexError(index)

        start = index * self.stride
        input_ids = self.token_tensor[start : start + self.max_length]
        target_ids = self.token_tensor[start + 1 : start + self.max_length + 1]
        return input_ids, target_ids


def create_dataloaders(
    train_token_tensor: torch.Tensor,
    val_token_tensor: torch.Tensor,
    *,
    context_length: int,
    stride: int,
    batch_size: int,
    num_workers: int,
    device: torch.device,
) -> tuple[DataLoader, DataLoader, CachedGPTDataset, CachedGPTDataset]:
    # === REF-MODEL CHANGE =============================================
    # Kein train_ratio-Split mehr. Beide Tensoren sind bereits eindeutig
    # als Referenz-Training bzw. originale M0-Validation definiert.
    train_dataset = CachedGPTDataset(
        train_token_tensor,
        max_length=context_length,
        stride=stride,
    )
    val_dataset = CachedGPTDataset(
        val_token_tensor,
        max_length=context_length,
        stride=stride,
    )
    # =================================================================

    common_loader_args = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": device.type == "cuda",
        "persistent_workers": num_workers > 0,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        drop_last=True,
        **common_loader_args,
    )
    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )

    return train_loader, val_loader, train_dataset, val_dataset


(
    TRAIN_LOADER,
    VAL_LOADER,
    TRAIN_DATASET,
    VAL_DATASET,
) = create_dataloaders(
    TRAIN_TOKEN_TENSOR,
    VAL_TOKEN_TENSOR,
    context_length=MODEL_CONFIG["context_length"],
    stride=DATA_CONFIG["stride"],
    batch_size=DATA_CONFIG["batch_size"],
    num_workers=DATA_CONFIG["num_workers"],
    device=DEVICE,
)

print(f"Trainingsfenster:   {len(TRAIN_DATASET):,}")
print(f"Validierungsfenster: {len(VAL_DATASET):,}")
print(f"Trainingsschritte pro Epoche: {len(TRAIN_LOADER):,}")
print(
    "Verarbeitete Tokenpositionen pro Epoche: "
    f"{len(TRAIN_LOADER) * DATA_CONFIG['batch_size'] * MODEL_CONFIG['context_length']:,}"
)


### 6.1 Validation-Sanity-Check


In [ ]:
# === REF-MODEL CHANGE =================================================
# Die ursprüngliche M0-Validation wird bewusst NICHT neu gespeichert.
# Diese Zelle ist nur ein zusätzlicher Sanity-Check.
print("Originaler Validation-Cache:")
print(PATH_CONFIG["original_val_tokens"])
print(f"Validation-Tokenanzahl: {len(VAL_TOKEN_TENSOR):,}")
print(f"Validation-Fenster:     {len(VAL_DATASET):,}")
print(
    "Identisch mit Suffix des modifizierten Gesamtkorpus:",
    torch.equal(
        TOKEN_TENSOR[-len(VAL_TOKEN_TENSOR):],
        VAL_TOKEN_TENSOR,
    ),
)
# =====================================================================


## 7. GPT-Modellarchitektur

Die Architektur entspricht der für $M_0$ verwendeten GPT-Implementierung.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        num_heads: int,
        qkv_bias: bool = False,
    ) -> None:
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out muss durch num_heads teilbar sein.")

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, _ = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        queries = queries.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        values = values.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        causal_mask = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask, -torch.inf)

        attention_weights = torch.softmax(
            attention_scores / math.sqrt(self.head_dim),
            dim=-1,
        )
        attention_weights = self.dropout(attention_weights)

        context = (attention_weights @ values).transpose(1, 2)
        context = context.reshape(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized + self.shift


class GELU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 0.5 * x * (
            1.0
            + torch.tanh(
                math.sqrt(2.0 / math.pi)
                * (x + 0.044715 * x.pow(3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
        )
        self.pos_emb = nn.Embedding(
            cfg["context_length"],
            cfg["emb_dim"],
        )
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[
                TransformerBlock(cfg)
                for _ in range(cfg["n_layers"])
            ]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False,
        )

    def forward(self, in_idx: torch.Tensor) -> torch.Tensor:
        _, sequence_length = in_idx.shape
        if sequence_length > self.pos_emb.num_embeddings:
            raise ValueError(
                f"Sequenzlänge {sequence_length} überschreitet "
                f"die Context Length {self.pos_emb.num_embeddings}."
            )

        token_embeddings = self.tok_emb(in_idx)
        position_ids = torch.arange(
            sequence_length,
            device=in_idx.device,
        )
        position_embeddings = self.pos_emb(position_ids)

        x = token_embeddings + position_embeddings
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


## 8. Loss und Evaluation


In [ ]:
def calc_loss_batch(
    input_batch: torch.Tensor,
    target_batch: torch.Tensor,
    model: nn.Module,
    device: torch.device,
) -> torch.Tensor:
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)

    logits = model(input_batch)
    return F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
    )


@torch.no_grad()
def calc_loss_loader(
    data_loader: DataLoader,
    model: nn.Module,
    device: torch.device,
    num_batches: int | None = None,
) -> float:
    if len(data_loader) == 0:
        return float("nan")

    batches_to_evaluate = (
        len(data_loader)
        if num_batches is None
        else min(num_batches, len(data_loader))
    )

    total_loss = 0.0
    for batch_index, (input_batch, target_batch) in enumerate(data_loader):
        if batch_index >= batches_to_evaluate:
            break
        loss = calc_loss_batch(
            input_batch,
            target_batch,
            model,
            device,
        )
        total_loss += loss.item()

    return total_loss / batches_to_evaluate


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    eval_batches: int,
) -> tuple[float, float]:
    was_training = model.training
    model.eval()

    train_loss = calc_loss_loader(
        train_loader,
        model,
        device,
        num_batches=eval_batches,
    )
    val_loss = calc_loss_loader(
        val_loader,
        model,
        device,
        num_batches=eval_batches,
    )

    model.train(was_training)
    return train_loss, val_loss


## 9. Textgenerierung für Trainingsstichproben

Während des Trainings werden in festen Abständen deterministische
Textstichproben erzeugt.


In [ ]:
@torch.no_grad()
def generate_text(
    model: nn.Module,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int,
    context_size: int,
    temperature: float = 0.0,
    top_k: int | None = None,
) -> str:
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens darf nicht negativ sein.")
    if temperature < 0:
        raise ValueError("temperature darf nicht negativ sein.")

    token_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=next(model.parameters()).device,
    ).unsqueeze(0)

    was_training = model.training
    model.eval()

    for _ in range(max_new_tokens):
        conditioned_ids = token_ids[:, -context_size:]
        logits = model(conditioned_ids)[:, -1, :]

        if temperature == 0:
            next_token = torch.argmax(
                logits,
                dim=-1,
                keepdim=True,
            )
        else:
            logits = logits / max(temperature, 1e-8)

            if top_k is not None:
                effective_top_k = min(top_k, logits.shape[-1])
                top_logits, top_indices = torch.topk(
                    logits,
                    effective_top_k,
                    dim=-1,
                )
                probabilities = torch.softmax(top_logits, dim=-1)
                sampled_position = torch.multinomial(
                    probabilities,
                    num_samples=1,
                )
                next_token = torch.gather(
                    top_indices,
                    dim=-1,
                    index=sampled_position,
                )
            else:
                probabilities = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(
                    probabilities,
                    num_samples=1,
                )

        token_ids = torch.cat((token_ids, next_token), dim=1)

    model.train(was_training)
    return tokenizer.decode(token_ids.squeeze(0).tolist())


## 10. Checkpoints und Lernraten-Scheduler

Checkpoints enthalten Modell-, Optimizer- und Scheduler-Zustand sowie
Global Step, Tokenzahl und Loss-Historie.


In [ ]:
class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        *,
        total_steps: int,
        warmup_steps: int,
        initial_lr: float,
        peak_lr: float,
        min_lr: float,
    ) -> None:
        if total_steps <= 0:
            raise ValueError("total_steps muss positiv sein.")
        if warmup_steps < 0:
            raise ValueError("warmup_steps darf nicht negativ sein.")
        if warmup_steps >= total_steps:
            warmup_steps = max(0, total_steps - 1)
        if not 0 <= min_lr <= peak_lr:
            raise ValueError("Es muss 0 <= min_lr <= peak_lr gelten.")

        self.optimizer = optimizer
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.initial_lr = initial_lr
        self.peak_lr = peak_lr
        self.min_lr = min_lr
        self.step_number = 0

        self._set_lr(self.learning_rate_at(0))

    def learning_rate_at(self, step_number: int) -> float:
        step_number = max(0, step_number)

        if self.warmup_steps > 0 and step_number <= self.warmup_steps:
            fraction = step_number / self.warmup_steps
            return self.initial_lr + fraction * (
                self.peak_lr - self.initial_lr
            )

        denominator = max(1, self.total_steps - self.warmup_steps)
        progress = (
            step_number - self.warmup_steps
        ) / denominator
        progress = min(max(progress, 0.0), 1.0)

        cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + (
            self.peak_lr - self.min_lr
        ) * cosine_factor

    def _set_lr(self, learning_rate: float) -> None:
        for parameter_group in self.optimizer.param_groups:
            parameter_group["lr"] = learning_rate

    def step(self) -> float:
        self.step_number += 1
        learning_rate = self.learning_rate_at(self.step_number)
        self._set_lr(learning_rate)
        return learning_rate

    def state_dict(self) -> dict[str, Any]:
        return {
            "total_steps": self.total_steps,
            "warmup_steps": self.warmup_steps,
            "initial_lr": self.initial_lr,
            "peak_lr": self.peak_lr,
            "min_lr": self.min_lr,
            "step_number": self.step_number,
        }

    def load_state_dict(self, state: dict[str, Any]) -> None:
        self.total_steps = int(state["total_steps"])
        self.warmup_steps = int(state["warmup_steps"])
        self.initial_lr = float(state["initial_lr"])
        self.peak_lr = float(state["peak_lr"])
        self.min_lr = float(state["min_lr"])
        self.step_number = int(state["step_number"])
        self._set_lr(self.learning_rate_at(self.step_number))


def empty_history() -> dict[str, list[float]]:
    return {
        "global_step": [],
        "tokens_seen": [],
        "train_loss": [],
        "val_loss": [],
        "learning_rate": [],
    }


def default_training_state() -> dict[str, int]:
    return {
        "global_step": 0,
        "tokens_seen": 0,
        "completed_epochs": 0,
    }


def move_optimizer_state_to_device(
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def load_training_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    *,
    device: torch.device,
    fallback_tokens_per_step: int,
) -> tuple[dict[str, int], dict[str, list[float]], dict[str, Any]]:
    checkpoint = torch_load_compatible(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        model.load_state_dict(checkpoint["model_state_dict"])

        if (
            optimizer is not None
            and "optimizer_state_dict" in checkpoint
        ):
            optimizer.load_state_dict(
                checkpoint["optimizer_state_dict"]
            )
            move_optimizer_state_to_device(optimizer, device)

        state = default_training_state()
        state.update(checkpoint.get("training_state", {}))
        state["global_step"] = int(
            checkpoint.get(
                "global_step",
                state.get("global_step", 0),
            )
        )

        if state.get("tokens_seen", 0) == 0 and state["global_step"] > 0:
            state["tokens_seen"] = (
                state["global_step"] * fallback_tokens_per_step
            )
            print(
                "Hinweis: Der alte Checkpoint enthält keine Tokenzahl. "
                "Sie wurde aus global_step angenähert."
            )

        history = checkpoint.get("history", empty_history())
        return state, history, checkpoint

    if not isinstance(checkpoint, dict):
        raise TypeError("Der Checkpoint besitzt ein unbekanntes Format.")

    # Rückwärtskompatibilität mit einem reinen model.state_dict().
    model.load_state_dict(checkpoint)
    return default_training_state(), empty_history(), {}


def save_training_checkpoint(
    checkpoint_path: Path,
    *,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    training_state: dict[str, int],
    history: dict[str, list[float]],
    model_config: dict[str, Any],
) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "format_version": 2,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "training_state": dict(training_state),
        "global_step": int(training_state["global_step"]),
        "history": history,
        "model_config": model_config,
    }
    torch.save(checkpoint, checkpoint_path)


def create_scheduler(
    optimizer: torch.optim.Optimizer,
    *,
    scheduler_total_steps: int,
    checkpoint: dict[str, Any],
    resumed: bool,
) -> WarmupCosineScheduler:
    saved_scheduler_state = checkpoint.get("scheduler_state_dict")

    if saved_scheduler_state is not None:
        scheduler = WarmupCosineScheduler(
            optimizer,
            total_steps=max(1, int(saved_scheduler_state["total_steps"])),
            warmup_steps=int(saved_scheduler_state["warmup_steps"]),
            initial_lr=float(saved_scheduler_state["initial_lr"]),
            peak_lr=float(saved_scheduler_state["peak_lr"]),
            min_lr=float(saved_scheduler_state["min_lr"]),
        )
        scheduler.load_state_dict(saved_scheduler_state)

        remaining_steps = max(
            0,
            scheduler.total_steps - scheduler.step_number,
        )
        print(
            "Scheduler-Zustand wiederhergestellt. "
            f"Verbleibende geplante Schritte: {remaining_steps:,}"
        )
        # === REF-MODEL CHANGE =========================================
        # Bei einem Resume ist der Scheduler-Zustand des Reference-Runs
        # maßgeblich. Der konfigurierte Plan wird nur zur Plausibilität
        # geprüft.
        if scheduler_total_steps != scheduler.total_steps:
            print(
                "Hinweis: TRAIN_CONFIG['scheduler_total_steps'] weicht "
                "vom gespeicherten Scheduler-Plan ab. Für den Resume wird "
                "der gespeicherte Scheduler-Zustand verwendet."
            )
        # ================================================================
        return scheduler

    if resumed:
        print(
            "Alter Checkpoint ohne Scheduler-Zustand: "
            "Es beginnt eine neue LR-Phase."
        )
        peak_lr = TRAIN_CONFIG["legacy_resume_peak_lr"]
        warmup_steps = TRAIN_CONFIG["legacy_resume_warmup_steps"]
        initial_lr = peak_lr if warmup_steps == 0 else min(
            TRAIN_CONFIG["initial_lr"],
            peak_lr,
        )
    else:
        peak_lr = TRAIN_CONFIG["peak_lr"]
        warmup_steps = TRAIN_CONFIG["warmup_steps"]
        initial_lr = TRAIN_CONFIG["initial_lr"]

    return WarmupCosineScheduler(
        optimizer,
        # === REF-MODEL CHANGE =========================================
        total_steps=max(1, scheduler_total_steps),
        # ================================================================
        warmup_steps=warmup_steps,
        initial_lr=initial_lr,
        peak_lr=peak_lr,
        min_lr=TRAIN_CONFIG["min_lr"],
    )


## 11. Trainingsfunktion

Das Referenztraining läuft über vollständige Dataset-Durchläufe, bis exakt
`target_global_step` erreicht ist. Dadurch erhält $M_{ref}$ dasselbe
Optimizer-Update-Budget wie der für die Experimente verwendete Zustand von
$M_0$.


In [ ]:
def print_training_sample(
    model: nn.Module,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int,
) -> None:
    sample = generate_text(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        context_size=model.pos_emb.num_embeddings,
        temperature=0.0,
    )
    print("\n--- Deterministische Stichprobe ---")
    print(sample)
    print("------------------------------------\n")


def train_model(
    *,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    device: torch.device,
    tokenizer,
    target_global_step: int,
    output_dir: Path,
    training_state: dict[str, int],
    history: dict[str, list[float]],
) -> tuple[dict[str, list[float]], dict[str, int]]:
    # === REF-MODEL CHANGE =============================================
    # Das Stopkriterium ist jetzt der exakte globale Optimizer-Step und
    # nicht mehr eine vorgegebene Anzahl Epochen.
    if target_global_step <= 0:
        raise ValueError("target_global_step muss positiv sein.")

    if training_state["global_step"] >= target_global_step:
        print(
            "Ziel-Step bereits erreicht: "
            f"{training_state['global_step']:,} >= "
            f"{target_global_step:,}"
        )
        return history, training_state
    # =================================================================

    try:
        while training_state["global_step"] < target_global_step:
            displayed_epoch = training_state["completed_epochs"] + 1
            print(
                f"Starte Dataset-Durchlauf {displayed_epoch} "
                f"(Global Step {training_state['global_step']:,} / "
                f"{target_global_step:,}) ..."
            )

            model.train()
            epoch_completed = True

            for input_batch, target_batch in train_loader:
                # Exakt am Ziel stoppen, auch wenn ein Dataset-Durchlauf
                # dadurch nur teilweise verarbeitet wird.
                if training_state["global_step"] >= target_global_step:
                    epoch_completed = False
                    break

                learning_rate = scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                loss = calc_loss_batch(
                    input_batch,
                    target_batch,
                    model,
                    device,
                )
                loss.backward()

                gradient_clip = TRAIN_CONFIG["gradient_clip_norm"]
                if gradient_clip is not None:
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=gradient_clip,
                    )

                optimizer.step()

                training_state["global_step"] += 1
                training_state["tokens_seen"] += input_batch.numel()
                global_step = training_state["global_step"]

                if global_step % TRAIN_CONFIG["eval_every_steps"] == 0:
                    train_loss, val_loss = evaluate_model(
                        model,
                        train_loader,
                        val_loader,
                        device,
                        TRAIN_CONFIG["eval_batches"],
                    )

                    history["global_step"].append(global_step)
                    history["tokens_seen"].append(
                        training_state["tokens_seen"]
                    )
                    history["train_loss"].append(train_loss)
                    history["val_loss"].append(val_loss)
                    history["learning_rate"].append(learning_rate)

                    print(
                        f"Ep {displayed_epoch} "
                        f"(Step {global_step:07d}): "
                        f"Train {train_loss:.3f}, "
                        f"Val {val_loss:.3f}, "
                        f"LR {learning_rate:.2e}"
                    )

                if (
                    TRAIN_CONFIG["sample_every_steps"] > 0
                    and global_step
                    % TRAIN_CONFIG["sample_every_steps"]
                    == 0
                ):
                    print_training_sample(
                        model,
                        tokenizer,
                        TRAIN_CONFIG["sample_prompt"],
                        max_new_tokens=TRAIN_CONFIG["sample_tokens"],
                    )

                if (
                    TRAIN_CONFIG["checkpoint_every_steps"] > 0
                    and global_step
                    % TRAIN_CONFIG["checkpoint_every_steps"]
                    == 0
                ):
                    checkpoint_path = (
                        output_dir
                        / f"model_step_{global_step:07d}.pth"
                    )
                    save_training_checkpoint(
                        checkpoint_path,
                        model=model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        training_state=training_state,
                        history=history,
                        model_config=MODEL_CONFIG,
                    )
                    print(f"Checkpoint gespeichert: {checkpoint_path}")

                # === REF-MODEL CHANGE =================================
                # Stop unmittelbar nach dem Update, das exakt den
                # gewünschten Global Step erreicht hat.
                if global_step >= target_global_step:
                    epoch_completed = False
                    break
                # =======================================================

            if not epoch_completed:
                print(
                    f"Exakter Ziel-Step erreicht: "
                    f"{training_state['global_step']:,}"
                )
                break

            training_state["completed_epochs"] += 1

            print_training_sample(
                model,
                tokenizer,
                TRAIN_CONFIG["sample_prompt"],
                max_new_tokens=TRAIN_CONFIG["sample_tokens"],
            )

            epoch_checkpoint = (
                output_dir
                / (
                    f"model_epoch_"
                    f"{training_state['completed_epochs']:03d}_"
                    f"step_{training_state['global_step']:07d}.pth"
                )
            )
            save_training_checkpoint(
                epoch_checkpoint,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                training_state=training_state,
                history=history,
                model_config=MODEL_CONFIG,
            )
            print(f"Epochen-Checkpoint gespeichert: {epoch_checkpoint}")

    except KeyboardInterrupt:
        interrupted_path = (
            output_dir
            / (
                f"model_step_"
                f"{training_state['global_step']:07d}_interrupted.pth"
            )
        )
        save_training_checkpoint(
            interrupted_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            training_state=training_state,
            history=history,
            model_config=MODEL_CONFIG,
        )
        print(f"Unterbrechungs-Checkpoint gespeichert: {interrupted_path}")

    return history, training_state


## 12. Modell und Optimizer initialisieren oder Checkpoint laden

Für einen neuen Lauf wird der bekannte Seed unmittelbar vor der
Modellinitialisierung erneut gesetzt. Ein vorhandener Referenz-Checkpoint kann
über `PATH_CONFIG["resume_checkpoint"]` fortgesetzt werden.


In [ ]:
# === REF-MODEL CHANGE =================================================
# Seed unmittelbar vor der Modellinitialisierung setzen.
# Ohne Step-0-Checkpoint ist dies die sauberste verfügbare Rekonstruktion
# der ursprünglichen Initialisierungsprozedur.
set_seed(TRAIN_CONFIG["seed"])
# =====================================================================

MODEL = GPTModel(MODEL_CONFIG).to(DEVICE)
OPTIMIZER = torch.optim.AdamW(
    MODEL.parameters(),
    lr=TRAIN_CONFIG["peak_lr"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

TRAINING_STATE = default_training_state()
HISTORY = empty_history()
LOADED_CHECKPOINT: dict[str, Any] = {}
RESUMED = False

resume_path = PATH_CONFIG["resume_checkpoint"]
if resume_path is not None and resume_path.exists():
    print(f"Lade Checkpoint: {resume_path}")
    (
        TRAINING_STATE,
        HISTORY,
        LOADED_CHECKPOINT,
    ) = load_training_checkpoint(
        resume_path,
        MODEL,
        OPTIMIZER,
        device=DEVICE,
        fallback_tokens_per_step=(
            DATA_CONFIG["batch_size"]
            * MODEL_CONFIG["context_length"]
        ),
    )
    RESUMED = True
    print(
        f"Checkpoint geladen. Global Step: "
        f"{TRAINING_STATE['global_step']:,}"
    )
else:
    if resume_path is not None:
        raise FileNotFoundError(
            "Konfigurierter Resume-Checkpoint wurde nicht gefunden: "
            f"{resume_path}"
        )
    print("Neues Referenzmodell wird von Grund auf trainiert.")

# === REF-MODEL CHANGE =================================================
TARGET_GLOBAL_STEP = TRAIN_CONFIG["target_global_step"]
SCHEDULER_TOTAL_STEPS = TRAIN_CONFIG["scheduler_total_steps"]

if TARGET_GLOBAL_STEP > SCHEDULER_TOTAL_STEPS:
    raise ValueError(
        "target_global_step darf für diesen Vergleich nicht größer als "
        "scheduler_total_steps sein."
    )

SCHEDULER = create_scheduler(
    OPTIMIZER,
    scheduler_total_steps=SCHEDULER_TOTAL_STEPS,
    checkpoint=LOADED_CHECKPOINT,
    resumed=RESUMED,
)

remaining_updates = max(
    0,
    TARGET_GLOBAL_STEP - TRAINING_STATE["global_step"],
)
# =====================================================================

parameter_count = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)

print(f"Modellparameter: {parameter_count:,}")
print(f"Aktueller Global Step: {TRAINING_STATE['global_step']:,}")
print(f"Ziel-Global-Step:      {TARGET_GLOBAL_STEP:,}")
print(f"Noch auszuführende Updates: {remaining_updates:,}")
print(f"Scheduler-Gesamtplan: {SCHEDULER_TOTAL_STEPS:,}")
print(
    "Aktuelle Lernrate:",
    OPTIMIZER.param_groups[0]["lr"],
)


## 13. Training ausführen

`RUN_TRAINING` bleibt standardmäßig `False`. Nach erfolgreicher Prüfung von
Referenzkorpus und Validation kann der Wert in der Konfigurationszelle für den
eigentlichen Lauf auf `True` gesetzt werden.


In [ ]:
if RUN_TRAINING:
    HISTORY, TRAINING_STATE = train_model(
        model=MODEL,
        train_loader=TRAIN_LOADER,
        val_loader=VAL_LOADER,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        device=DEVICE,
        tokenizer=TOKENIZER,
        # === REF-MODEL CHANGE =========================================
        target_global_step=TRAIN_CONFIG["target_global_step"],
        # ==============================================================
        output_dir=PATH_CONFIG["output_dir"],
        training_state=TRAINING_STATE,
        history=HISTORY,
    )

    final_checkpoint = (
        PATH_CONFIG["output_dir"]
        / f"model_final_step_{TRAINING_STATE['global_step']:07d}.pth"
    )
    save_training_checkpoint(
        final_checkpoint,
        model=MODEL,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        training_state=TRAINING_STATE,
        history=HISTORY,
        model_config=MODEL_CONFIG,
    )
    print(f"Finaler Checkpoint gespeichert: {final_checkpoint}")
else:
    print(
        "Training ist deaktiviert. "
        "Prüfe zuerst Referenzkorpus und Validation-Sanity-Checks. "
        "Setze danach RUN_TRAINING = True und führe die Konfigurations-, "
        "Initialisierungs- und Trainingszelle erneut aus."
    )


## 14. Loss-Verlauf darstellen

Die Darstellung verwendet die während des Referenztrainings gespeicherte
Loss-Historie.


In [ ]:
def plot_training_history(
    history: dict[str, list[float]],
    *,
    save_path: Path | None = None,
) -> None:
    if not history["global_step"]:
        print("Noch keine Evaluationswerte in der Historie vorhanden.")
        return

    figure, axis = plt.subplots(figsize=(8, 4.5))
    axis.plot(
        history["global_step"],
        history["train_loss"],
        label="Training Loss",
    )
    axis.plot(
        history["global_step"],
        history["val_loss"],
        linestyle="-.",
        label="Validation Loss",
    )
    axis.set_xlabel("Globaler Optimizer-Schritt")
    axis.set_ylabel("Cross-Entropy-Loss")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_path, bbox_inches="tight")
        print(f"Loss-Plot gespeichert: {save_path}")

    plt.show()


plot_training_history(
    HISTORY,
    save_path=PATH_CONFIG["output_dir"] / "loss_plot.pdf",
)


## Ergebnis

Der finale Referenz-Checkpoint wird unter
`reference_span_v1/model_final_step_0046460.pth` gespeichert. Weiterführende
Fakten- und Vergleichsauswertungen erfolgen getrennt in der Ergebnisanalyse.
